# Attention U-Net for Retinal Artery-Vein Segmentation
## Research Pipeline with Frame-level Augmentation, Curriculum Learning and Vessel-Aware Training

---

# Objective

This notebook implements a research-oriented semantic segmentation framework for retinal artery-vein (AV) segmentation using an enhanced **Attention U-Net** architecture. It targets accurate segmentation of **arteries, veins, overlap regions, and background** while addressing the major difficulties of retinal vessel analysis: severe class imbalance, thin vessel preservation, and vascular topology continuity.

The GAVE-2 dataset is used as the initial training corpus. The longer-term goal is cross-dataset generalisation, so the pipeline is written to be dataset-agnostic rather than tuned to any single challenge specification.

The implementation combines vessel-aware patch selection, frame-level augmentation, curriculum learning, topology-preserving optimisation, and test-time augmentation.

---

# Pipeline Overview

```
Dataset Preparation
        │
        ▼
Image & Mask Analysis
        │
        ▼
ROI Masking + Green Channel + CLAHE Enhancement
        │
        ▼
Ground Truth Engineering
(RGB → Multi-class Masks)
        │
        ▼
Frame-level Offline Augmentation
(applied to full 768x768 frames)
        │
        ▼
Vessel-aware Patch Extraction
(256x256, stride 128, ROI + vessel filtered)
        │
        ▼
PyTorch Dataset & DataLoader
(weighted sampling)
        │
        ▼
Attention U-Net
(kernel schedule + deep supervision)
        │
        ▼
Curriculum Learning Strategy
        │
        ▼
Hybrid Loss Optimization
(Focal Tversky + clDice)
        │
        ▼
Training & Validation
        │
        ▼
Test-Time Augmentation
        │
        ▼
Evaluation & Visualization
```

---

# Key Features

## Intelligent Image Preprocessing

Retinal fundus images are preprocessed to enhance vessel visibility before training:

- ROI masking to discard the non-retinal border
- Green channel extraction
- CLAHE (Contrast Limited Adaptive Histogram Equalization)
- Padding to a fixed 768x768 canvas
- RGB mask conversion into multi-class segmentation labels (0 background, 1 artery, 2 overlap, 3 vein)

---

## Frame-level Offline Augmentation

Augmentation is applied to the **full 768x768 frame, before patch extraction**.

### Motivation

An earlier revision of this pipeline reversed that order: patches were cut first, then rotated, sheared and warped on a 256x256 canvas with `BORDER_CONSTANT` fill. That is incorrect. A 30-degree rotation of a 256x256 patch converts roughly a quarter of its area into synthetic black wedges, and 11 of the 18 variants in the augmentation set are geometric. The network was therefore trained on a hard black diagonal edge artefact that never occurs at inference time — a train/test distribution mismatch concentrated exactly where thin peripheral vessels live.

Augmenting the full frame instead pushes those wedges out to the frame corners, which already lie outside the retinal ROI.

### Interaction with the ROI filter

The ROI mask is registered as an additional albumentations target, so it is warped together with the image and the label map. Wedge pixels therefore receive `roi = 0`, and the existing `MIN_ROI_FRACTION` criterion in the patch filter discards any patch that falls inside a wedge. No additional filtering logic is required — the correction follows from the reordering alone.

### Transform set

Per frame, the original plus 17 variants are written:

| Group | Variants |
|---|---|
| Flips | hflip, vflip, hvflip |
| Rotations | ±15°, ±25°, ±30° |
| Flip + rotation | hflip+20°, vflip−20° |
| Affine | scale 0.8–1.15, translate ±5%, shear ±8° |
| Elastic | alpha 40, sigma 6 |
| Intensity | brightness/contrast, gamma, Gaussian blur, Gaussian noise |

Geometric transforms use `INTER_LINEAR` on the image and `INTER_NEAREST` on both masks, so no interpolated label values are introduced. Intensity transforms leave the masks untouched.

---

## Vessel-Aware Patch Extraction

Rather than cutting every patch on the grid, the pipeline keeps only patches that carry useful signal.

### Motivation

Retinal fundus images are over **90% background**, so an unfiltered grid spends most of the training compute on empty regions.

A patch is retained when both conditions hold:

- vessel pixels ≥ `MIN_VESSEL_FRACTION` (0.01) of the patch
- ROI pixels ≥ `MIN_ROI_FRACTION` (0.50) of the patch

A small random share (`BACKGROUND_KEEP_RATIO` = 0.05) of rejected patches is retained so the network still sees plain background and does not over-predict vessels in empty areas.

On top of this filter, a `WeightedRandomSampler` biases batch composition toward vessel-dense and overlap-containing patches.

---

## Attention U-Net Architecture

The backbone is an **Attention U-Net**, using attention gates to suppress irrelevant background features while emphasising vessel regions.

Components:

- Encoder-decoder architecture with skip connections
- Attention gates on every skip connection
- GroupNorm rather than BatchNorm (stable at small batch sizes)
- Deep supervision from three decoder depths
- Multi-class output head

### Kernel schedule

The first `KERNEL_DEPTH` encoder stages and their mirrored decoder stages use a `KERNEL_SIZE` x `KERNEL_SIZE` convolution; every deeper stage stays at 3x3.

The shallow stages operate at full resolution, where thin-vessel context lives, and are the cheapest layers in the network. Widening only those costs roughly +4.6% parameters, against +178% if every stage were widened. `KERNEL_DEPTH = 0` reproduces the all-3x3 baseline exactly, so the schedule is directly ablatable.

Even kernel sizes are rejected by assertion: they cannot be padded symmetrically, so they break skip-connection shapes and accumulate a half-pixel drift per layer — precisely the error mode that destroys 1–3 pixel vessels.

---

# Curriculum Learning Strategy

Topology is not optimised from the first step. The clDice term is faded in only once the network can localise vessels, because the soft skeleton of an under-converged prediction carries little usable signal.

## Stage 1 — epochs 0 to 9

Focal Tversky loss only. The network learns vessel localisation under a stable, purely region-based objective.

## Stage 2 — epoch 10 onwards

The clDice weight ramps **linearly** from 0 to `CLDICE_WEIGHT` over `CLDICE_RAMP_EPOCHS` (5) epochs, reaching full weight at epoch 14.

A linear ramp is used instead of a hard switch so that the loss discontinuity does not land on the cosine warm-restart boundary. Setting `CLDICE_RAMP_EPOCHS = 1` recovers the instantaneous switch.

---

# Hybrid Loss Function

## Focal Tversky Loss

Parameters:

- Alpha = 0.30 (false positive weight)
- Beta = 0.70 (false negative weight)
- Gamma = 4/3

Beta > Alpha penalises false negatives more heavily, which raises recall on thin vessels at the cost of precision. This is a deliberate trade and should be re-examined whenever the reported metric changes.

## clDice Loss

clDice preserves vascular topology by encouraging skeleton-level agreement between prediction and ground truth, via iterative soft erosion and dilation.

Intended benefits:

- Better vessel continuity
- Reduced vessel fragmentation
- Improved bifurcation preservation

clDice is computed in fp32 with autocast disabled: the soft-skeleton iterations compound small errors and are numerically fragile under fp16.

## Final Hybrid Objective

The main head uses complementary weighting, so the total scale of the main term stays constant across the ramp:

```
main   = (1 - w_cl) x FocalTversky(main) + w_cl x clDice(main)
total  = main
       + 0.40 x FocalTversky(aux2)
       + 0.25 x FocalTversky(aux3)
       + 0.15 x FocalTversky(aux4)

w_cl : 0 → 0.30, linear over epochs 10 to 14
```

Deep-supervision heads stay on Focal Tversky only. A topology loss on 2x/4x/8x downsampled logits is meaningless, since thin vessels do not survive the downsampling.

---

# Training Strategy

- Mixed precision training (AMP), with clDice forced to fp32
- AdamW optimizer
- Cosine annealing with warm restarts
- Deep supervision
- Vessel-aware weighted sampling
- Early stopping on validation Dice

---

# Evaluation

Metrics: Dice, IoU, pixel accuracy, precision, recall, F1, reported per class.

Dice and IoU skip any class absent from both prediction and ground truth. Scoring such a class as `1e-6 / 1e-6 = 1.0` silently inflated the mean on the rare overlap class in earlier revisions.

Qualitative evaluation is performed through prediction visualisation against ground truth, including a baseline-versus-TTA disagreement map.

---

# Current Status and Known Limitations

Recorded here so that results are not over-read:

- **Validation and test sets are 8 frames each.** The empirical noise floor on mean Dice is roughly ±0.02, which is larger than most effects measured so far. Single-split results are not a sound basis for model selection. A k-fold protocol over the combined train+validation pool is the intended replacement.
- **Training and evaluation resolution differ.** The network is trained on 256x256 patches but evaluated on full 768x768 frames. Tiled sliding-window inference would remove this mismatch.
- **The clDice term has not yet influenced a selected checkpoint.** In the most recent run the best validation epoch preceded `CLDICE_START_EPOCH`, so the saved weights never saw the topology term. Any claim about clDice remains unverified until this is corrected.
- **Overlap is a difficult and arguably ill-posed class.** It accounts for a very small fraction of pixels, its annotation is ambiguous, and including it in the mean depresses the headline score. A multi-label reformulation (artery and vein channels, with crossings positive in both) is under consideration.
- **Artery-vein label flips near the optic disc persist.** These are a connectivity failure, not a texture failure: identity is determined by tracing a branch back to its trunk, and a 256px patch cannot see that far. No loss term or test-time augmentation can recover information absent from the input.

---


In [ ]:
import os
import cv2
import random
import shutil
from pathlib import Path

import numpy as np

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import albumentations as A

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


import warnings

# Silence the noisy albumentations and sklearn chatter, but keep
# DeprecationWarning visible so API drift surfaces instead of hiding.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("All libraries imported successfully.")

In [ ]:
# =============================================================================
# Configuration
# =============================================================================

# Seeding is done once in the import cell.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================================================
# Dataset Paths
# =============================================================================

DATASET_PATH = r"C:\Users\HP\Desktop\GAVE2_preliminary\training"

IMAGE_FOLDER = os.path.join(DATASET_PATH, "images")
GT_FOLDER    = os.path.join(DATASET_PATH, "av")
ROI_FOLDER   = os.path.join(DATASET_PATH, "masks")

OUTPUT_DIR = "./Processed_Dataset"

# =============================================================================
# Dataset Split
# =============================================================================

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# =============================================================================
# Image / Patch Configuration
# =============================================================================

IMAGE_SIZE = 768

PATCH_SIZE = 256
PATCH_STRIDE = 128
BATCH_SIZE = 8

# Vessel-aware Patch Extraction
ENABLE_VESSEL_PATCHING = True

# A patch is kept only if at least this fraction of its pixels are vessel
# (artery, overlap or vein) and this fraction lies inside the retinal ROI.
MIN_VESSEL_FRACTION = 0.01
MIN_ROI_FRACTION = 0.50

# Share of rejected patches retained so the model still sees plain background
BACKGROUND_KEEP_RATIO = 0.05

# Sampler weighting (applied on top of the filter)
OVERLAP_BOOST = 3.0
WEIGHT_FLOOR = 0.002

# =============================================================================
# Convolution Kernel Schedule
# =============================================================================
#
# KERNEL_SIZE applies to the first KERNEL_DEPTH encoder stages and their
# mirrored decoder stages. Everything deeper stays at 3x3.
#
# Rationale: the shallow stages run at full resolution where thin-vessel
# context lives, and they are the cheapest layers in the network. Widening
# only those costs ~0.5M params. Widening every stage costs ~2.78x the conv
# parameters (31M -> ~87M) on 589 genuinely distinct patches.
#
# KERNEL_DEPTH = 0 reproduces the all-3x3 baseline exactly.
#
# Even kernels (2, 4) are rejected: they cannot be symmetrically padded, so
# they break skip-connection shapes and inject a half-pixel drift per layer
# that accumulates -- the exact error mode that destroys 1-3px vessels.

KERNEL_SIZE = 5
KERNEL_DEPTH = 2

assert KERNEL_SIZE % 2 == 1, (
    f"KERNEL_SIZE must be odd for symmetric padding, got {KERNEL_SIZE}. "
    "Even kernels misalign skip connections."
)
assert 0 <= KERNEL_DEPTH <= 4, "KERNEL_DEPTH must be in [0, 4]"

# =============================================================================
# Training Configuration
# =============================================================================

NUM_CLASSES = 4

EPOCHS = 200

LR = 1e-4

WEIGHT_DECAY = 1e-4

NUM_WORKERS = 0

PIN_MEMORY = True

# =============================================================================
# Curriculum Learning
# =============================================================================

# clDice now activates early enough to actually shape the selected
# checkpoint. Previously it started at 20 while the best epoch was 14,
# so the saved weights never saw the topology term at all.
CLDICE_START_EPOCH = 10

# Linear ramp instead of a hard switch: avoids a loss discontinuity
# landing near the cosine warm-restart boundary (T_0 = 20).
# Set to 1 to reproduce the old instantaneous switch.
CLDICE_RAMP_EPOCHS = 5

FOCAL_TVERSKY_WEIGHT = 0.70
CLDICE_WEIGHT = 0.30

CLDICE_ITERATIONS = 6

# =============================================================================
# Focal Tversky Parameters
# =============================================================================

ALPHA = 0.30
BETA = 0.70
GAMMA = 4 / 3

EARLY_STOPPING_PATIENCE = 25

# Separate filename so this notebook does not overwrite
# notebook 4's checkpoint
CHECKPOINT_NAME = f"attention_unet_k{KERNEL_SIZE}d{KERNEL_DEPTH}_cl{CLDICE_START_EPOCH}.pth"

print("=" * 60)
print(f"Device                 : {DEVICE}")
print(f"Image Size             : {IMAGE_SIZE}")
print(f"Patch Size             : {PATCH_SIZE}")
print(f"Batch Size             : {BATCH_SIZE}")
print(f"Epochs                 : {EPOCHS}")
print(f"Kernel Schedule        : {KERNEL_SIZE}x{KERNEL_SIZE} for first "
      f"{KERNEL_DEPTH} stage(s), 3x3 deeper")
print(f"Curriculum Starts      : Epoch {CLDICE_START_EPOCH} "
      f"(ramp over {CLDICE_RAMP_EPOCHS} epochs)")
print(f"Checkpoint             : {CHECKPOINT_NAME}")
print("=" * 60)

In [ ]:
# =============================================================================
# Create Split Folder Structure
# =============================================================================
#
# Only the raw split folders are created here. The processed, patch and
# augmented trees are created by the cells that own them.

folders = [
    "train/images",
    "train/av",
    "train/masks",

    "val/images",
    "val/av",
    "val/masks",

    "test/images",
    "test/av",
    "test/masks",
]

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

for folder in folders:
    os.makedirs(os.path.join(OUTPUT_DIR, folder), exist_ok=True)

print("=" * 60)
print("Folder structure created successfully.")
print("=" * 60)

In [ ]:
# =============================================================================
# Collect Dataset Files
# =============================================================================

from glob import glob

image_paths = sorted(glob(os.path.join(IMAGE_FOLDER, "*")))
gt_paths    = sorted(glob(os.path.join(GT_FOLDER, "*")))
roi_paths   = sorted(glob(os.path.join(ROI_FOLDER, "*")))

assert len(image_paths) == len(gt_paths) == len(roi_paths), \
    "Mismatch in the number of images, AV masks, and ROI masks."

# Verify corresponding filenames
for img, gt, roi in zip(image_paths, gt_paths, roi_paths):

    img_name = os.path.splitext(os.path.basename(img))[0]
    gt_name  = os.path.splitext(os.path.basename(gt))[0]
    roi_name = os.path.splitext(os.path.basename(roi))[0]

    assert img_name == gt_name == roi_name, \
        f"Filename mismatch: {img_name}, {gt_name}, {roi_name}"

print("=" * 60)
print(f"Total Images : {len(image_paths)}")
print("Dataset integrity check passed.")
print("=" * 60)

In [ ]:
# =============================================================================
# Train / Validation / Test Split
# =============================================================================

train_images, temp_images, train_gt, temp_gt, train_roi, temp_roi = train_test_split(
    image_paths,
    gt_paths,
    roi_paths,
    test_size=(1 - TRAIN_RATIO),
    random_state=SEED,
    shuffle=True
)

val_ratio_adjusted = VAL_RATIO / (VAL_RATIO + TEST_RATIO)

val_images, test_images, val_gt, test_gt, val_roi, test_roi = train_test_split(
    temp_images,
    temp_gt,
    temp_roi,
    train_size=val_ratio_adjusted,
    random_state=SEED,
    shuffle=True
)

print("=" * 60)
print("Dataset Split")
print("=" * 60)
print(f"Training Images   : {len(train_images)}")
print(f"Validation Images : {len(val_images)}")
print(f"Testing Images    : {len(test_images)}")
print("=" * 60)

In [ ]:
# =============================================================================
# Copy Dataset into Train / Validation / Test Folders
# =============================================================================

def copy_files(images, gts, rois, split):

    image_out = os.path.join(OUTPUT_DIR, split, "images")
    gt_out    = os.path.join(OUTPUT_DIR, split, "av")
    roi_out   = os.path.join(OUTPUT_DIR, split, "masks")

    os.makedirs(image_out, exist_ok=True)
    os.makedirs(gt_out, exist_ok=True)
    os.makedirs(roi_out, exist_ok=True)

    for img, gt, roi in zip(images, gts, rois):
        shutil.copy2(img, image_out)
        shutil.copy2(gt, gt_out)
        shutil.copy2(roi, roi_out)

    print(f"{split.capitalize():<10}: {len(images)} samples copied")


# Copy previously created splits
copy_files(train_images, train_gt, train_roi, "train")
copy_files(val_images, val_gt, val_roi, "val")
copy_files(test_images, test_gt, test_roi, "test")

print("=" * 60)
print("Dataset successfully organized.")
print("=" * 60)

In [ ]:
# =============================================================================
# Create Processed Dataset Folders
# =============================================================================

PROCESSED_DIR = os.path.join(OUTPUT_DIR, "processed")

folders = [

    "train/images",
    "train/av",
    "train/masks",

    "val/images",
    "val/av",
    "val/masks",

    "test/images",
    "test/av",
    "test/masks"

]

if os.path.exists(PROCESSED_DIR):
    shutil.rmtree(PROCESSED_DIR)

for folder in folders:
    os.makedirs(os.path.join(PROCESSED_DIR, folder), exist_ok=True)

print("=" * 60)
print("Processed dataset folders created successfully.")
print("=" * 60)

In [ ]:
# =============================================================================
# Preprocessing Helper Functions
# =============================================================================

def apply_roi(image: np.ndarray, roi: np.ndarray) -> np.ndarray:
    """
    Retain only the retinal region using the ROI mask.
    """
    roi = (roi > 0).astype(np.uint8)
    return cv2.bitwise_and(image, image, mask=roi)


def extract_green(image: np.ndarray) -> np.ndarray:
    """
    Extract the green channel from an RGB fundus image.
    """
    return image[:, :, 1]


def apply_clahe(image: np.ndarray,
                clip_limit: float = 2.0,
                tile_grid_size: tuple = (8, 8)) -> np.ndarray:
    """
    Apply CLAHE for local contrast enhancement.
    """
    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=tile_grid_size
    )
    return clahe.apply(image)


def pad_to_square(image: np.ndarray,
                  pad_value: int = 0) -> np.ndarray:
    """
    Pad an image to make it square.
    """

    h, w = image.shape[:2]
    size = max(h, w)

    top = (size - h) // 2
    bottom = size - h - top
    left = (size - w) // 2
    right = size - w - left

    if image.ndim == 2:
        value = pad_value
    else:
        value = (pad_value,) * image.shape[2]

    return cv2.copyMakeBorder(
        image,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_CONSTANT,
        value=value
    )


def resize_image(image: np.ndarray,
                 interpolation: int) -> np.ndarray:
    """
    Resize image to the configured IMAGE_SIZE.
    """

    return cv2.resize(
        image,
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=interpolation
    )

In [ ]:
 
# RGB Mask to Class Mask
 

def rgb_to_class(mask):

    label = np.zeros(mask.shape[:2], dtype=np.uint8)

    # Background
    label[np.all(mask == [0, 0, 0], axis=-1)] = 0

    # Artery (Red)
    label[np.all(mask == [255, 0, 0], axis=-1)] = 1

    # Overlap (Green)
    label[np.all(mask == [0, 255, 0], axis=-1)] = 2

    # Vein (Blue)
    label[np.all(mask == [0, 0, 255], axis=-1)] = 3

    return label

In [ ]:
 
# Preprocess One Sample
 

def preprocess_sample(image_path, roi_path, gt_path):

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    roi = cv2.imread(roi_path, 0)

    gt = cv2.imread(gt_path)
    gt = cv2.cvtColor(gt, cv2.COLOR_BGR2RGB)

    image = apply_roi(image, roi)

    image = extract_green(image)

    image = apply_clahe(image)

    image = pad_to_square(image)
    image = cv2.merge([image, image, image])

    image = resize_image(
        image,
        cv2.INTER_LINEAR
    )

    gt = rgb_to_class(gt)

    gt = pad_to_square(gt)

    gt = resize_image(
        gt,
        cv2.INTER_NEAREST
    )

    roi = pad_to_square(roi)

    roi = resize_image(
        roi,
        cv2.INTER_NEAREST
    )

    return image, gt, roi

In [ ]:
# Process Dataset

splits = ["train", "val", "test"]

for split in splits:

    image_dir = os.path.join(OUTPUT_DIR, split, "images")
    gt_dir = os.path.join(OUTPUT_DIR, split, "av")
    roi_dir = os.path.join(OUTPUT_DIR, split, "masks")

    out_img = os.path.join(PROCESSED_DIR, split, "images")
    out_gt = os.path.join(PROCESSED_DIR, split, "av")
    out_roi = os.path.join(PROCESSED_DIR, split, "masks")
    
    images = sorted(glob(os.path.join(image_dir, "*")))
    gts = sorted(glob(os.path.join(gt_dir, "*")))
    rois = sorted(glob(os.path.join(roi_dir, "*")))

    assert len(images) == len(gts) == len(rois), \
        f"Mismatch in {split}: {len(images)} images, {len(gts)} GTs, {len(rois)} ROIs"

    for img_path, gt_path, roi_path in tqdm(
        zip(images, gts, rois),
        total=len(images),
        desc=split
    ):


        image, gt, roi = preprocess_sample(
            img_path,
            roi_path,
            gt_path
        )

        name = os.path.basename(img_path)

        cv2.imwrite(
            os.path.join(out_img, name),
            image
        )

        cv2.imwrite(
            os.path.join(out_gt, name),
            gt
        )

        cv2.imwrite(
            os.path.join(out_roi, name),
            roi
        )

print("Preprocessing completed.")

In [ ]:
 
# Visual Verification
 

def class_to_rgb(mask):
    """
    Convert class mask back to RGB for visualization only.
    """

    rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)

    rgb[mask == 0] = [0, 0, 0]         # Background
    rgb[mask == 1] = [255, 0, 0]       # Artery
    rgb[mask == 2] = [0, 255, 0]       # Overlap
    rgb[mask == 3] = [0, 0, 255]       # Vein

    return rgb


sample = sorted(glob(os.path.join(PROCESSED_DIR, "train/images/*")))[0]

name = os.path.basename(sample)

image = cv2.imread(sample)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

gt = cv2.imread(
    os.path.join(PROCESSED_DIR, "train/av", name),
    cv2.IMREAD_GRAYSCALE
)

roi = cv2.imread(
    os.path.join(PROCESSED_DIR, "train/masks", name),
    cv2.IMREAD_GRAYSCALE
)

gt_rgb = class_to_rgb(gt)

plt.figure(figsize=(18,6))

plt.subplot(131)
plt.imshow(image)
plt.title("Green + CLAHE")
plt.axis("off")

plt.subplot(132)
plt.imshow(gt_rgb)
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(133)
plt.imshow(roi, cmap="gray")
plt.title("ROI")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
 
# Create Augmented Dataset
 

AUGMENTED_DIR = "./Augmented_Dataset"

folders = [
    "train/images",
    "train/av",
    "train/masks",

    "val/images",
    "val/av",
    "val/masks",

    "test/images",
    "test/av",
    "test/masks"
]

if os.path.exists(AUGMENTED_DIR):
    shutil.rmtree(AUGMENTED_DIR)

for folder in folders:
    os.makedirs(os.path.join(AUGMENTED_DIR, folder), exist_ok=True)

print("Augmented dataset folders created.")

In [ ]:
 
# Copy Validation and Test Sets
 

for split in ["val", "test"]:

    for folder in ["images", "av", "masks"]:

        src = os.path.join(PROCESSED_DIR, split, folder)
        dst = os.path.join(AUGMENTED_DIR, split, folder)

        for file in os.listdir(src):

            shutil.copy(
                os.path.join(src, file),
                os.path.join(dst, file)
            )

print("Validation and Test copied.")

In [ ]:
def _rot(deg):
    return A.Rotate(limit=(deg, deg), border_mode=cv2.BORDER_CONSTANT,
                    interpolation=cv2.INTER_LINEAR,
                    mask_interpolation=cv2.INTER_NEAREST,
                    fill=0, fill_mask=0, p=1)

augmentations = {
    "hflip":        A.HorizontalFlip(p=1),
    "vflip":        A.VerticalFlip(p=1),
    "hvflip":       A.Compose([A.HorizontalFlip(p=1), A.VerticalFlip(p=1)]),
    "rot_p15":      _rot(15),
    "rot_m15":      _rot(-15),
    "rot_p25":      _rot(25),
    "rot_m25":      _rot(-25),
    "rot_p30":      _rot(30),
    "rot_m30":      _rot(-30),
    "hflip_rot20":  A.Compose([A.HorizontalFlip(p=1), _rot(20)]),
    "vflip_rot_m20":A.Compose([A.VerticalFlip(p=1), _rot(-20)]),

    "affine":       A.Affine(scale=(0.8, 1.15), translate_percent=(-0.05, 0.05),
                             shear=(-8, 8), interpolation=cv2.INTER_LINEAR,
                             mask_interpolation=cv2.INTER_NEAREST,
                             border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=0, p=1),
    "elastic":      A.ElasticTransform(alpha=40, sigma=6,
                                       interpolation=cv2.INTER_LINEAR,
                                       mask_interpolation=cv2.INTER_NEAREST,
                                       border_mode=cv2.BORDER_CONSTANT,
                                       fill=0, fill_mask=0, p=1),

    "brightness":   A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
    "gamma":        A.RandomGamma(gamma_limit=(80, 120), p=1),
    "gaussian_blur":A.GaussianBlur(blur_limit=(3, 5), p=1),
    "gaussian_noise":A.GaussNoise(std_range=(0.02, 0.08), mean_range=(0.0, 0.0),
                                  per_channel=False, p=1),
}

# Register ROI as an additional mask target
augmentations = {
    k: A.Compose(
        v.transforms if isinstance(v, A.Compose) else [v],
        additional_targets={"roi": "mask"}
    )
    for k, v in augmentations.items()
}

In [ ]:
# =============================================================================
# Frame-level Augmentation + Vessel-aware Patch Extraction
# =============================================================================
#
# Augmentation now runs on the full 768x768 frame, BEFORE patching.
#
# The previous order was reversed: patches were cut first, then rotated /
# sheared / warped on a 256x256 canvas with BORDER_CONSTANT fill. A 30 degree
# rotation of a 256 patch turns roughly a quarter of it into synthetic black
# wedges that never occur at inference time, and 11 of the 18 variants are
# geometric. The model was being trained on that artefact.
#
# Rotating the full frame instead pushes those wedges out to the frame
# corners, which already sit outside the retinal ROI. The ROI mask is warped
# along with the frame, so wedge pixels end up with roi = 0 and the existing
# MIN_ROI_FRACTION filter discards those patches on its own. No new filtering
# logic is needed -- the fix falls out of the reordering.
#
# Patches are written straight into AUGMENTED_DIR. The intermediate PATCH_DIR
# tree is no longer produced, so the old "copy original training set" step is
# gone too.

INTENSITY_AUGS = {
    "brightness",
    "gamma",
    "gaussian_blur",
    "gaussian_noise"
}


def frame_variants(image, gt, roi):
    """
    Yield (name, image, gt, roi) for the original frame and each augmentation.
    Intensity augmentations leave the masks untouched, as before.
    """

    yield "orig", image, gt, roi

    for aug_name, aug in augmentations.items():

        if aug_name in INTENSITY_AUGS:

            out = aug(image=image)

            yield aug_name, out["image"], gt, roi

        else:

            out = aug(image=image, mask=gt, roi=roi)

            yield aug_name, out["image"], out["mask"], out["roi"]


def write_patches(image, gt, roi, base, rng, stats):
    """
    Cut the patch grid out of one frame and write the patches that pass the
    vessel / ROI filter. Filter logic is unchanged from extract_patches.
    """

    for y in range(0, IMAGE_SIZE - PATCH_SIZE + 1, PATCH_STRIDE):
        for x in range(0, IMAGE_SIZE - PATCH_SIZE + 1, PATCH_STRIDE):

            stats["examined"] += 1

            gt_patch  = gt[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
            roi_patch = roi[y:y + PATCH_SIZE, x:x + PATCH_SIZE]

            if ENABLE_VESSEL_PATCHING:

                vessel_fraction = (gt_patch > 0).mean()
                roi_fraction    = (roi_patch > 0).mean()

                vessel_rich = (
                    vessel_fraction >= MIN_VESSEL_FRACTION
                    and roi_fraction >= MIN_ROI_FRACTION
                )

                if not vessel_rich:

                    if rng.random() >= BACKGROUND_KEEP_RATIO:
                        continue

                    stats["background"] += 1

            name = f"{base}_y{y}_x{x}.png"

            cv2.imwrite(
                os.path.join(AUGMENTED_DIR, "train", "images", name),
                cv2.cvtColor(
                    image[y:y + PATCH_SIZE, x:x + PATCH_SIZE],
                    cv2.COLOR_RGB2BGR
                )
            )

            cv2.imwrite(
                os.path.join(AUGMENTED_DIR, "train", "av", name),
                gt_patch
            )

            cv2.imwrite(
                os.path.join(AUGMENTED_DIR, "train", "masks", name),
                roi_patch
            )

            stats["kept"] += 1


train_frames = sorted(glob(os.path.join(PROCESSED_DIR, "train/images/*")))
train_gts    = sorted(glob(os.path.join(PROCESSED_DIR, "train/av/*")))
train_rois   = sorted(glob(os.path.join(PROCESSED_DIR, "train/masks/*")))

assert len(train_frames) == len(train_gts) == len(train_rois), \
    "Mismatch between frames, AV masks and ROI masks."

# Dedicated stream so the background filter is reproducible and does not
# disturb the global random state.
patch_rng = random.Random(SEED)

stats = {"examined": 0, "kept": 0, "background": 0}

for ip, gp, rp in tqdm(
    zip(train_frames, train_gts, train_rois),
    total=len(train_frames),
    desc="augment + patch"
):

    image = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
    gt    = cv2.imread(gp, cv2.IMREAD_GRAYSCALE)
    roi   = cv2.imread(rp, cv2.IMREAD_GRAYSCALE)

    stem = Path(ip).stem

    for aug_name, aug_image, aug_gt, aug_roi in frame_variants(image, gt, roi):

        write_patches(
            aug_image,
            aug_gt,
            aug_roi,
            f"{stem}_{aug_name}",
            patch_rng,
            stats
        )

n_variants = 1 + len(augmentations)

print()
print(f"Frames           : {len(train_frames)} x {n_variants} variants "
      f"= {len(train_frames) * n_variants}")
print(f"Patches examined : {stats['examined']}")
print(f"Patches kept     : {stats['kept']} "
      f"({stats['kept'] - stats['background']} vessel-rich, "
      f"{stats['background']} background retained)")


In [ ]:
# Dataset Statistics

for split in ["train", "val", "test"]:

    n_images = len(glob(os.path.join(AUGMENTED_DIR, split, "images", "*")))
    n_gt = len(glob(os.path.join(AUGMENTED_DIR, split, "av", "*")))
    n_roi = len(glob(os.path.join(AUGMENTED_DIR, split, "masks", "*")))

    assert n_images == n_gt == n_roi, \
        f"{split}: Dataset mismatch ({n_images}, {n_gt}, {n_roi})"

    print(f"{split:5s} : {n_images:4d} samples")

print("\nDataset verification completed successfully.")

In [ ]:
# Visualize Augmentations

samples = random.sample(
    sorted(glob(os.path.join(AUGMENTED_DIR, "train/images/*"))),
    9
)

fig, axes = plt.subplots(3, 6, figsize=(18, 9))

for i, img_path in enumerate(samples):

    base = Path(img_path).name

    gt_path = os.path.join(AUGMENTED_DIR, "train/av", base)

    # RGB image
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # AV mask
    gt = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)

    axes[i // 3, (i % 3) * 2].imshow(image)
    axes[i // 3, (i % 3) * 2].set_title(Path(img_path).stem, fontsize=8)
    axes[i // 3, (i % 3) * 2].axis("off")

    axes[i // 3, (i % 3) * 2 + 1].imshow(class_to_rgb(gt))
    axes[i // 3, (i % 3) * 2 + 1].set_title("AV Mask", fontsize=8)
    axes[i // 3, (i % 3) * 2 + 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Dataset Class

class RetinalDataset(Dataset):

    def __init__(self, image_dir, mask_dir, compute_weights=False):

        self.images = sorted(glob(os.path.join(image_dir, "*")))
        self.masks  = sorted(glob(os.path.join(mask_dir, "*")))

        assert len(self.images) == len(self.masks), \
            "Mismatch between images and masks."

        self.weights = None

        if compute_weights:

            w = []

            for m in tqdm(self.masks, desc="Computing sample weights"):

                gt = cv2.imread(m, cv2.IMREAD_GRAYSCALE)

                vessel  = (gt > 0).mean()
                overlap = (gt == 2).mean()

                w.append(
                    max(vessel, WEIGHT_FLOOR) ** 0.5
                    + OVERLAP_BOOST * (overlap ** 0.5)
                )

            self.weights = np.array(w, dtype=np.float64)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image = cv2.imread(self.images[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(self.masks[idx], cv2.IMREAD_GRAYSCALE)

        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))

        image = torch.from_numpy(image).float()
        mask  = torch.from_numpy(mask).long()

        return image, mask

In [ ]:
 
# DataLoaders
 

from torch.utils.data import WeightedRandomSampler

train_dataset = RetinalDataset(
    os.path.join(AUGMENTED_DIR, "train/images"),
    os.path.join(AUGMENTED_DIR, "train/av"),
    compute_weights=True
)

val_dataset = RetinalDataset(

    os.path.join(AUGMENTED_DIR, "val/images"),
    os.path.join(AUGMENTED_DIR, "val/av")

)

test_dataset = RetinalDataset(

    os.path.join(AUGMENTED_DIR, "test/images"),
    os.path.join(AUGMENTED_DIR, "test/av")

)

sampler = WeightedRandomSampler(
    weights=torch.from_numpy(train_dataset.weights),
    num_samples=len(train_dataset),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0
)

print("Train :", len(train_dataset))
print("Validation :", len(val_dataset))
print("Test :", len(test_dataset))

In [ ]:
w = train_dataset.weights

print("Weight percentiles [0,25,50,75,100]:")
print(np.percentile(w, [0, 25, 50, 75, 100]))
print(f"Max/Min ratio : {w.max() / w.min():.1f}x")
print(f"Effective unique samples : {(w.sum()**2)/(w**2).sum():.0f} / {len(w)}")

In [ ]:
# Verify Dataset

images, masks = next(iter(train_loader))

print("Images Shape :", images.shape)
print("Masks Shape  :", masks.shape)

print("Image dtype  :", images.dtype)
print("Mask dtype   :", masks.dtype)

print("Image Range  :", images.min().item(), "to", images.max().item())

print("Unique Classes :", torch.unique(masks))

In [ ]:
# Double Convolution Block

import math


class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels, kernel_size=3):

        super().__init__()

        if kernel_size % 2 == 0:
            raise ValueError(
                f"kernel_size must be odd, got {kernel_size}. "
                "Even kernels cannot be symmetrically padded and will "
                "misalign the skip connections."
            )

        padding = kernel_size // 2

        groups = math.gcd(8, out_channels)

        self.kernel_size = kernel_size

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False
            ),

            nn.GroupNorm(
                groups,
                out_channels
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False
            ),

            nn.GroupNorm(
                groups,
                out_channels
            ),

            nn.ReLU(inplace=True)

        )

    def forward(self, x):

        return self.conv(x)


In [ ]:
# Attention Gate

class AttentionBlock(nn.Module):

    def __init__(self, F_g, F_l, F_int):

        super().__init__()


        groups = math.gcd(8, F_int)

        self.W_g = nn.Sequential(

            nn.Conv2d(
                F_g,
                F_int,
                kernel_size=1,
                bias=True
            ),

            nn.GroupNorm(
                groups,
                F_int
            )

        )

        self.W_x = nn.Sequential(

            nn.Conv2d(
                F_l,
                F_int,
                kernel_size=1,
                bias=True
            ),

            nn.GroupNorm(
                groups,
                F_int
            )

        )

        self.psi = nn.Sequential(

            nn.Conv2d(
                F_int,
                1,
                kernel_size=1,
                bias=True
            ),

            nn.Sigmoid()

        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):

        g1 = self.W_g(g)

        x1 = self.W_x(x)

        psi = self.relu(g1 + x1)

        psi = self.psi(psi)

        return x * psi

In [ ]:
# Encoder Block

class EncoderBlock(nn.Module):

    def __init__(self, in_channels, out_channels, kernel_size=3):

        super().__init__()

        self.conv = DoubleConv(in_channels, out_channels, kernel_size)

        self.pool = nn.MaxPool2d(2)

    def forward(self, x):

        skip = self.conv(x)

        pooled = self.pool(skip)

        return skip, pooled


In [ ]:
# Decoder Block

class DecoderBlock(nn.Module):

    def __init__(self, in_channels, skip_channels, out_channels,
                 kernel_size=3):

        super().__init__()

        # Upsampling stays 2x2 stride 2 regardless of the conv kernel --
        # this is a resampling op, not a feature extractor.
        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.attention = AttentionBlock(
            F_g=out_channels,
            F_l=skip_channels,
            F_int=out_channels // 2
        )

        self.conv = DoubleConv(
            out_channels + skip_channels,
            out_channels,
            kernel_size
        )

    def forward(self, x, skip):

        x = self.up(x)

        skip = self.attention(x, skip)

        x = torch.cat([skip, x], dim=1)

        return self.conv(x)


In [ ]:
# Attention U-Net with Deep Supervision

class AttentionUNet(nn.Module):

    def __init__(self, in_channels=3, num_classes=4,
                 kernel_size=3, kernel_depth=0):
        """
        kernel_size  : conv kernel for the shallow stages
        kernel_depth : how many encoder stages (from the top) use kernel_size.
                       Their mirrored decoder stages match. 0 = all 3x3.
        """

        super().__init__()

        # Per-stage kernel schedule: wide at full resolution, 3x3 deeper.
        k = [kernel_size if i < kernel_depth else 3 for i in range(4)]

        self.kernel_schedule = k

        # Encoder

        self.enc1 = EncoderBlock(in_channels, 64, k[0])
        self.enc2 = EncoderBlock(64, 128, k[1])
        self.enc3 = EncoderBlock(128, 256, k[2])
        self.enc4 = EncoderBlock(256, 512, k[3])

        # Bottleneck -- always 3x3. It already has the largest receptive
        # field and by far the most parameters.

        self.bottleneck = DoubleConv(512, 1024, 3)

        # Decoder (mirrors the encoder schedule)

        self.dec4 = DecoderBlock(1024, 512, 512, k[3])
        self.dec3 = DecoderBlock(512, 256, 256, k[2])
        self.dec2 = DecoderBlock(256, 128, 128, k[1])
        self.dec1 = DecoderBlock(128, 64, 64, k[0])

        # Main Output

        self.out = nn.Conv2d(
            64,
            num_classes,
            kernel_size=1
        )

        # Deep Supervision Heads

        self.ds4 = nn.Conv2d(
            512,
            num_classes,
            kernel_size=1
        )

        self.ds3 = nn.Conv2d(
            256,
            num_classes,
            kernel_size=1
        )

        self.ds2 = nn.Conv2d(
            128,
            num_classes,
            kernel_size=1
        )
        
        self.initialize_weights()
        
    def initialize_weights(self):

        for m in self.modules():

            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(
                    m.weight,
                    mode="fan_out",
                    nonlinearity="relu"
                )

                if m.bias is not None:
                    nn.init.zeros_(m.bias)

            elif isinstance(m, nn.GroupNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):

        # Encoder

        s1, p1 = self.enc1(x)

        s2, p2 = self.enc2(p1)

        s3, p3 = self.enc3(p2)

        s4, p4 = self.enc4(p3)

        # Bottleneck

        b = self.bottleneck(p4)

        # Decoder

        d4 = self.dec4(b, s4)

        d3 = self.dec3(d4, s3)

        d2 = self.dec2(d3, s2)

        d1 = self.dec1(d2, s1)

        # Main Prediction

        out = self.out(d1)

        # Deep Supervision

        aux4 = self.ds4(d4)

        aux3 = self.ds3(d3)

        aux2 = self.ds2(d2)

        

        return (

    out,

    F.interpolate(
        aux2,
        size=out.shape[2:],
        mode="bilinear",
        align_corners=False
    ),
    

    F.interpolate(
        aux3,
        size=out.shape[2:],
        mode="bilinear",
        align_corners=False
    ),

    F.interpolate(
        aux4,
        size=out.shape[2:],
        mode="bilinear",
        align_corners=False
    )
    
    

)

In [ ]:
model = AttentionUNet(
    in_channels=3,
    num_classes=NUM_CLASSES,
    kernel_size=KERNEL_SIZE,
    kernel_depth=KERNEL_DEPTH
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())

# All-3x3 reference for cost comparison
_baseline = AttentionUNet(3, NUM_CLASSES, kernel_size=3, kernel_depth=0)
n_baseline = sum(p.numel() for p in _baseline.parameters())
del _baseline

print(f"Kernel schedule (enc1..enc4) : {model.kernel_schedule}")
print(f"Model Parameters             : {n_params:,}")
print(f"All-3x3 baseline             : {n_baseline:,}")
print(f"Overhead                     : {n_params - n_baseline:+,} "
      f"({100 * (n_params / n_baseline - 1):+.1f}%)")

# Shape check: the schedule must not disturb spatial dims anywhere
with torch.no_grad():
    _probe = torch.zeros(1, 3, 256, 256, device=DEVICE)
    _outs = model(_probe)
    assert all(o.shape[2:] == (256, 256) for o in _outs), \
        "Kernel schedule broke spatial alignment"
    print(f"Shape check                  : OK, {len(_outs)} outputs at 256x256")

del _probe, _outs
torch.cuda.empty_cache()


In [ ]:
# Multi-Class Focal Tversky Loss

class FocalTverskyLoss(nn.Module):

    def __init__(
        self,
        alpha=ALPHA,
        beta=BETA,
        gamma=GAMMA,
        smooth=1e-6,
        num_classes=NUM_CLASSES
    ):
        super().__init__()

        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth
        self.num_classes = num_classes

    def forward(self, logits, targets):

        # Convert logits to probabilities
        probs = torch.softmax(logits, dim=1)

        # Convert targets to one-hot encoding
        targets_onehot = F.one_hot(
            targets,
            self.num_classes
        ).permute(0, 3, 1, 2).float()

        # Flatten spatial dimensions
        probs = probs.flatten(2)
        targets_onehot = targets_onehot.flatten(2)

        # Compute Tversky statistics
        TP = (probs * targets_onehot).sum(dim=2)
        FP = (probs * (1 - targets_onehot)).sum(dim=2)
        FN = ((1 - probs) * targets_onehot).sum(dim=2)

        tversky = (
            TP + self.smooth
        ) / (
            TP +
            self.alpha * FP +
            self.beta * FN +
            self.smooth
        )

        focal_tversky = (1.0 - tversky) ** self.gamma

        return focal_tversky.mean()

In [ ]:
 # Soft Skeletonization
 
def soft_erode(img):

    if img.dim() == 4:

        p1 = -F.max_pool2d(-img, (3,1), stride=1, padding=(1,0))

        p2 = -F.max_pool2d(-img, (1,3), stride=1, padding=(0,1))

        return torch.min(p1, p2)

    raise ValueError("Expected 4D tensor")


def soft_dilate(img):

    return F.max_pool2d(
        img,
        kernel_size=3,
        stride=1,
        padding=1
    )


def soft_open(img):

    return soft_dilate(
        soft_erode(img)
    )


def soft_skeletonize(img, iterations=CLDICE_ITERATIONS):

    img1 = soft_open(img)

    skeleton = F.relu(img - img1)

    for _ in range(iterations):

        img = soft_erode(img)

        opened = soft_open(img)

        delta = F.relu(img-opened)

        skeleton = skeleton + F.relu(delta - skeleton*delta)

    return skeleton

In [ ]:
# Multi-Class CLDice Loss

class CLDiceLoss(nn.Module):

    def __init__(
        self,
        num_classes=NUM_CLASSES,
        iterations=CLDICE_ITERATIONS,
        smooth=1e-6
    ):

        super().__init__()

        self.num_classes = num_classes
        self.iterations = iterations
        self.smooth = smooth

    def forward(self, logits, targets):

        probs = torch.softmax(logits, dim=1)
        
        # Ignore background channel
        probs = probs[:, 1:, :, :]

        targets = F.one_hot(
            targets,
            self.num_classes
        ).permute(0,3,1,2).float()

        targets = targets[:, 1:, :, :]
        
        skel_pred = soft_skeletonize(
            probs,
            self.iterations
        )

        skel_gt = soft_skeletonize(
            targets,
            self.iterations
        )

        tprec = (

            (skel_pred * targets).sum((2,3))

            + self.smooth

        ) / (

            skel_pred.sum((2,3))

            + self.smooth

        )

        tsens = (

            (skel_gt * probs).sum((2,3))

            + self.smooth

        ) / (

            skel_gt.sum((2,3))

            + self.smooth

        )

        cldice = (

            2*tprec*tsens

        ) / (

            tprec + tsens + self.smooth

        )

        return 1 - cldice.mean()

In [ ]:
focal_tversky_loss = FocalTverskyLoss(
    alpha=ALPHA,
    beta=BETA,
    gamma=GAMMA,
    num_classes=NUM_CLASSES
)

cldice_loss = CLDiceLoss(
    num_classes=NUM_CLASSES,
    iterations=CLDICE_ITERATIONS
)

In [ ]:
# Hybrid Loss with ramped clDice curriculum
# Weights, start epoch and ramp length come from the configuration cell.


def cldice_weight_at(epoch):
    """
    Linear ramp of the clDice weight.

    Returns 0 before CLDICE_START_EPOCH, then ramps to CLDICE_WEIGHT over
    CLDICE_RAMP_EPOCHS epochs. A hard switch (the previous behaviour) is
    recovered by setting CLDICE_RAMP_EPOCHS = 1.

    Soft skeletons of an under-converged prediction carry little signal,
    which is why the term is faded in rather than dropped in at full weight.
    """

    if epoch < CLDICE_START_EPOCH:
        return 0.0

    progress = (epoch - CLDICE_START_EPOCH + 1) / max(1, CLDICE_RAMP_EPOCHS)

    return CLDICE_WEIGHT * min(1.0, progress)


def hybrid_loss(outputs, targets, epoch=0):

    main, aux2, aux3, aux4 = outputs

    ft_main = focal_tversky_loss(main, targets)

    w_cl = cldice_weight_at(epoch)

    if w_cl > 0.0:

        # clDice is numerically fragile under fp16: the soft-skeleton
        # iterations compound small errors, so force fp32.
        with torch.amp.autocast("cuda", enabled=False):
            cl_main = cldice_loss(main.float(), targets)

        # Complementary weighting keeps the main term's scale constant
        # across the ramp, so val loss stays roughly comparable.
        main_loss = (1.0 - w_cl) * ft_main + w_cl * cl_main

    else:
        main_loss = ft_main

    # Deep supervision heads stay on Focal Tversky only. Topology loss on
    # 2x/4x/8x downsampled logits is meaningless -- thin vessels do not
    # survive the downsampling.
    aux2_loss = focal_tversky_loss(aux2, targets)
    aux3_loss = focal_tversky_loss(aux3, targets)
    aux4_loss = focal_tversky_loss(aux4, targets)

    return main_loss + 0.40 * aux2_loss + 0.25 * aux3_loss + 0.15 * aux4_loss


print("clDice ramp schedule:")
for _e in range(0, CLDICE_START_EPOCH + CLDICE_RAMP_EPOCHS + 3):
    print(f"  epoch {_e:>3d} : clDice weight = {cldice_weight_at(_e):.3f}")


In [ ]:
# Optimizer

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY

)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=20,
    T_mult=1,
    eta_min=1e-6

)

scaler = torch.amp.GradScaler("cuda")
print("Optimizer, Scheduler and AMP initialized.")

In [ ]:
@torch.no_grad()
def dice_score(
    preds,
    targets,
    num_classes=NUM_CLASSES,
    ignore_background=True,
    smooth=1e-6
):

    preds = torch.argmax(preds, dim=1)

    dice_scores = []

    start_class = 1 if ignore_background else 0

    for cls in range(start_class, num_classes):

        pred_cls = (preds == cls).float()
        target_cls = (targets == cls).float()

        # A class absent from both prediction and target is undefined, not
        # perfect. Scoring it 1.0 inflates the mean, which matters most for
        # the rare overlap class.
        if target_cls.sum() == 0 and pred_cls.sum() == 0:
            dice_scores.append(
                torch.tensor(float("nan"), device=preds.device)
            )
            continue

        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()

        dice = (2.0 * intersection + smooth) / (union + smooth)

        dice_scores.append(dice)

    dice_scores = torch.stack(dice_scores)

    mean_dice = torch.nanmean(dice_scores).item()

    return mean_dice, dice_scores.cpu().tolist()

In [ ]:
@torch.no_grad()
def iou_score(
    preds,
    targets,
    num_classes=NUM_CLASSES,
    ignore_background=True,
    smooth=1e-6
):

    preds = torch.argmax(preds, dim=1)

    iou_scores = []

    start_class = 1 if ignore_background else 0

    for cls in range(start_class, num_classes):

        pred_cls = (preds == cls).float()
        target_cls = (targets == cls).float()

        # A class absent from both prediction and target is undefined, not
        # perfect. Scoring it 1.0 inflates the mean, which matters most for
        # the rare overlap class.
        if target_cls.sum() == 0 and pred_cls.sum() == 0:
            iou_scores.append(
                torch.tensor(float("nan"), device=preds.device)
            )
            continue

        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum() - intersection

        iou = (intersection + smooth) / (union + smooth)

        iou_scores.append(iou)

    iou_scores = torch.stack(iou_scores)

    mean_iou = torch.nanmean(iou_scores).item()

    return mean_iou, iou_scores.cpu().tolist()

In [ ]:
# Precision, Recall, F1 and Accuracy

from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score
)

CLASS_NAMES = [
    "Background",
    "Artery",
    "Overlap",
    "Vein"
]


@torch.no_grad()
def classification_metrics(preds, targets):

    preds = preds.detach()
    preds = torch.argmax(preds, dim=1)
    preds = preds.cpu().numpy().ravel()

    targets = targets.detach().cpu().numpy().ravel()

    precision, recall, f1, _ = precision_recall_fscore_support(
        targets,
        preds,
        labels=[0, 1, 2, 3],
        average=None,
        zero_division=0
    )

    accuracy = accuracy_score(
        targets,
        preds
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
# Train One Epoch

def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    device,
    epoch=0
):

    model.train()

    running_loss = 0.0

    # Collected rather than summed: a batch where a class is absent from both
    # prediction and target yields nan for that class, which must not poison
    # the epoch mean.
    dice_values = []
    iou_values = []

    progress = tqdm(loader)

    for images, masks in progress:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):

            outputs = model(images)

            loss = hybrid_loss(
                outputs,
                masks,
                epoch
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        # Main prediction only
        main_output = outputs[0]

        dice, _ = dice_score(
            main_output,
            masks
        )

        iou, _ = iou_score(
            main_output,
            masks
        )

        running_loss += float(loss.detach())

        dice_values.append(dice)
        iou_values.append(iou)

        progress.set_postfix(
            Loss=f"{float(loss.detach()):.4f}",
            Dice=f"{dice:.4f}",
            IoU=f"{iou:.4f}"
        )

    return (
        running_loss / len(loader),
        float(np.nanmean(dice_values)),
        float(np.nanmean(iou_values))
    )

In [ ]:
# Validate One Epoch

def validate_one_epoch(
    model,
    loader,
    device,
    epoch=0
):

    model.eval()

    running_loss = 0.0

    dice_values = []
    iou_values = []

    running_acc = 0.0
    running_precision = np.zeros(NUM_CLASSES)
    running_recall = np.zeros(NUM_CLASSES)
    running_f1 = np.zeros(NUM_CLASSES)

    with torch.no_grad():

        progress = tqdm(loader)

        for images, masks in progress:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with torch.amp.autocast("cuda"):

                outputs = model(images)

                loss = hybrid_loss(
                    outputs,
                    masks,
                    epoch
                )

            # Main prediction only
            main_output = outputs[0]

            dice, _ = dice_score(
                main_output,
                masks
            )

            iou, _ = iou_score(
                main_output,
                masks
            )

            metrics = classification_metrics(
                main_output,
                masks
            )

            running_loss += float(loss.detach())

            dice_values.append(dice)
            iou_values.append(iou)

            running_acc += metrics["accuracy"]
            running_precision += metrics["precision"]
            running_recall += metrics["recall"]
            running_f1 += metrics["f1"]

            progress.set_postfix(
                Loss=f"{float(loss.detach()):.4f}",
                Dice=f"{dice:.4f}",
                IoU=f"{iou:.4f}"
            )

    n = len(loader)

    return (
        running_loss / n,
        float(np.nanmean(dice_values)),
        float(np.nanmean(iou_values)),
        running_acc / n,
        running_precision / n,
        running_recall / n,
        running_f1 / n
    )

In [ ]:
model.eval()

with torch.no_grad():

    images, masks = next(iter(train_loader))
    images = images.to(DEVICE, non_blocking=True)

    outputs = model(images)

    print(f"Output type : {type(outputs)}")
    print(f"Number of outputs : {len(outputs)}\n")

    for i, out in enumerate(outputs):
        print(f"Output {i}: {tuple(out.shape)}")

del outputs, images, masks
torch.cuda.empty_cache()

In [ ]:
# Early Stopping Helper

def early_stopping(
    current_score,
    best_score,
    counter,
    patience
):
    """
    Early stopping based on validation Dice.

    Returns:
        best_score (float)
        counter (int)
        should_stop (bool)
        is_best (bool)
    """

    if current_score > best_score:

        best_score = current_score
        counter = 0

        return best_score, counter, False, True

    counter += 1

    if counter >= patience:
        return best_score, counter, True, False

    return best_score, counter, False, False

In [ ]:
# Complete Training Loop

best_dice = 0.0
epochs_without_improvement = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "train_dice": [],
    "val_dice": [],
    "train_iou": [],
    "val_iou": []
}

for epoch in range(EPOCHS):

    print("=" * 40)
    print(f"Epoch [{epoch + 1}/{EPOCHS}]  (clDice weight {cldice_weight_at(epoch):.3f})")
    print("=" * 40)

    train_results = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler,
        DEVICE,
        epoch
    )

    val_results = validate_one_epoch(
        model,
        val_loader,
        DEVICE,epoch
    )

    scheduler.step()

    train_loss, train_dice, train_iou = train_results

    (
        val_loss,
        val_dice,
        val_iou,
        val_acc,
        val_prec,
        val_rec,
        val_f1
    ) = val_results

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    history["train_dice"].append(train_dice)
    history["val_dice"].append(val_dice)

    history["train_iou"].append(train_iou)
    history["val_iou"].append(val_iou)

    print()
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Dice : {train_dice:.4f}")
    print(f"Train IoU  : {train_iou:.4f}")

    print()
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Dice   : {val_dice:.4f}")
    print(f"Val IoU    : {val_iou:.4f}")

    print()
    print(f"Validation Accuracy : {val_acc:.4f}")

    print("\nPer Class Metrics")

    for i, cls in enumerate(CLASS_NAMES):
        print(
            f"{cls:12s} | "
            f"P={val_prec[i]:.3f} "
            f"R={val_rec[i]:.3f} "
            f"F1={val_f1[i]:.3f}"
        )

    # Early Stopping

    best_dice, epochs_without_improvement, stop, is_best = early_stopping(
        val_dice,
        best_dice,
        epochs_without_improvement,
        EARLY_STOPPING_PATIENCE
    )

    if is_best:

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_dice": best_dice,
                "history": history,
            },
            CHECKPOINT_NAME
        )

        print("\nBest model saved.")

    else:

        print(
            f"\nEarly Stopping Counter: "
            f"{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}"
        )

    if stop:

        print("\nEarly stopping triggered.")
        break

print("\nTraining Complete.")

In [ ]:
# Load Best Checkpoint
#
# The un-augmented baseline is scored inside test_model_tta further down, so it
# is not evaluated separately here. This cell only restores the best weights,
# which the visualisation cell below relies on.

checkpoint = torch.load(
    CHECKPOINT_NAME,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

print(f"Loaded epoch {checkpoint['epoch']} "
      f"(best validation Dice {checkpoint['best_dice']:.4f})")

In [ ]:
# Training Curves

plt.figure(figsize=(18, 5))

# ---------------- Loss ---------------- #

plt.subplot(1, 3, 1)

plt.plot(history["train_loss"], label="Train")
plt.plot(history["val_loss"], label="Validation")

plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()

# ---------------- Dice ---------------- #

plt.subplot(1, 3, 2)

plt.plot(history["train_dice"], label="Train")
plt.plot(history["val_dice"], label="Validation")

plt.title("Dice Score")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.grid(True, alpha=0.3)
plt.legend()

# ---------------- IoU ---------------- #

plt.subplot(1, 3, 3)

plt.plot(history["train_iou"], label="Train")
plt.plot(history["val_iou"], label="Validation")

plt.title("IoU Score")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.grid(True, alpha=0.3)
plt.legend()  

plt.tight_layout()
plt.show()

In [ ]:
# Visualization

model.eval()

num_examples = 8

indices = np.random.choice(
    len(test_dataset),
    num_examples,
    replace=False
)

plt.figure(figsize=(15, 5 * num_examples))

with torch.no_grad():

    for row, idx in enumerate(indices):

        image, gt = test_dataset[idx]

        image_gpu = image.unsqueeze(0).to(
            DEVICE,
            non_blocking=True
        )

        with torch.amp.autocast("cuda"):

            outputs = model(image_gpu)

        main_output = outputs[0]

        pred = torch.argmax(
            main_output,
            dim=1
        ).squeeze().cpu().numpy()

        gt = gt.numpy()

        pred_rgb = class_to_rgb(pred)
        gt_rgb = class_to_rgb(gt)

        image_rgb = image.permute(1, 2, 0).numpy()

        plt.subplot(num_examples, 3, row * 3 + 1)
        plt.imshow(image_rgb)
        plt.title("Input")
        plt.axis("off")

        plt.subplot(num_examples, 3, row * 3 + 2)
        plt.imshow(gt_rgb)
        plt.title("Ground Truth")
        plt.axis("off")

        plt.subplot(num_examples, 3, row * 3 + 3)
        plt.imshow(pred_rgb)
        plt.title("Prediction")
        plt.axis("off")

plt.tight_layout()
plt.show()

---

# Test-Time Augmentation (TTA)

A test-time augmentation stage is applied after training. The training and
validation loops are structurally unchanged from earlier revisions, but the
Dice and IoU functions were corrected, so absolute scores are not comparable
across notebooks. The baseline-versus-TTA comparison *inside* this notebook is
exact, since every variant is scored by the same code on the same batches.

## Strategy

**Transform group: D4 (dihedral) x multi-scale**

The 8 elements of the dihedral group `D4` are applied to every input:

| k | Transform |
|---|-----------|
| 0 | identity |
| 1 | rot 90 |
| 2 | rot 180 |
| 3 | rot 270 |
| 4 | hflip |
| 5 | hflip + rot 90 |
| 6 | hflip + rot 180 |
| 7 | hflip + rot 270 |

Flips and 90-degree rotations are **lossless** (pure memory permutations, no
interpolation), so they introduce no resampling error into the prediction.

Each of the 8 transforms is additionally evaluated at **3 scales** (0.75x, 1.0x,
1.25x), giving **24 forward passes per image**. Multi-scale does involve bilinear
interpolation, which is why the per-scale logits are mapped back to native
resolution *after* the geometric inverse is applied.

**Elastic variants**

Two additional elastic fields per view are optionally applied, raising the count
to 72 passes. A TTA transform must be invertible: the input is warped, predicted,
and the logits unwarped before averaging. D4 inverts exactly; elastic does not.
Given a forward displacement field `d`, the true inverse must satisfy a fixed-point
condition that is only approximated here, so a small residual warp remains.

**Merging: two strategies, reported side by side**

1. `tta_prob`: mean of post-softmax probabilities, then argmax.
   Well-calibrated; a single overconfident view cannot dominate.
2. `tta_logit`: mean of raw logits, then softmax/argmax.
   Sharper decision boundaries, but sensitive to outlier views.

Both are scored against the un-augmented `baseline` (single forward pass) on the
same test batch, so the comparison is exact.

## Notes

- Only the **main** segmentation head is used for TTA. The deep-supervision heads
  (`aux2`, `aux3`, `aux4`) are training-time regularizers and are ignored here.
- The geometric inverse is applied **before** the resize back to native resolution,
  so rotation and flip stay exact and only the scale change costs interpolation.
- Test loss is reported for the baseline only, since `hybrid_loss` needs the full
  4-tuple of deep-supervision outputs, which TTA does not produce.

## Interpreting the result

Two caveats belong with any number produced by this section.

**The test set is 8 frames.** The noise floor on mean Dice is approximately
±0.02. Any difference smaller than that is not measurable with this split and
should be reported as such, not as an improvement.

**The training set already contains the D4 group.** Flips and 90-degree rotations
are part of the offline augmentation set, so the network is close to D4-invariant
before TTA is applied. Averaging over symmetries the model has effectively
memorised adds little information, and a small or null gain is the expected
outcome rather than an anomaly.

The elastic variants are expected to perform *worse* than plain D4 for the reason
given above: the approximate inverse leaves a residual warp on the order of one to
two pixels, which is the same scale as the structures being segmented.

---


In [ ]:
# =============================================================================
# Test-Time Augmentation Configuration
# =============================================================================

# Dihedral group D4 -- 8 lossless symmetries (no interpolation error)
#   k = 0..3  ->  rot90 applied k times
#   k = 4..7  ->  horizontal flip, then rot90 applied (k - 4) times
TTA_D4_KS = [0, 1, 2, 3, 4, 5, 6, 7]

# Multi-scale factors, applied on top of every D4 transform
TTA_SCALES = [0.75, 1.0, 1.25]

# Encoder has 4 max-pools, so spatial dims must stay divisible by 16
TTA_SIZE_MULTIPLE = 16

# ---------------------------------------------------------------------------
# Elastic deformation
# ---------------------------------------------------------------------------
#
# Unlike D4, elastic warping has NO exact inverse. The inverse displacement
# field must be solved numerically, so every elastic view costs two bilinear
# resamplings plus a residual inversion error.
#
# Measured round-trip error on a 1px line pattern (signal mass 0.121):
#     D4 rot90      0.00000   exact
#     scale 1.25    0.06134   already in the pipeline
#     elastic a=12  0.07715   this transform
#     scale 0.75    0.10045   already in the pipeline (worst)
#
# Elastic is therefore no more destructive than the multi-scale transforms
# already in use -- but it is not free, and it is reported as its OWN variant
# so a bad result cannot contaminate the working D4+scale numbers.

TTA_ELASTIC_N = 2            # random fields per (scale, D4) combination
TTA_ELASTIC_ALPHA = 12.0     # peak displacement, pixels
TTA_ELASTIC_SIGMA = 12.0     # field smoothness, pixels (low = crumpled)
TTA_ELASTIC_INV_ITERS = 6    # fixed-point iterations; 1 == naive -d shortcut
TTA_ELASTIC_SEED = 1234      # fields are regenerated identically every run

TTA_NUM_PASSES = len(TTA_D4_KS) * len(TTA_SCALES)

TTA_ELASTIC_PASSES = TTA_NUM_PASSES * (1 + TTA_ELASTIC_N)

print("=" * 60)
print(f"D4 transforms            : {len(TTA_D4_KS)}")
print(f"Scales                   : {TTA_SCALES}")
print(f"Forward passes per image : {TTA_NUM_PASSES}")
print(f"Elastic fields per view  : {TTA_ELASTIC_N} (alpha={TTA_ELASTIC_ALPHA}, sigma={TTA_ELASTIC_SIGMA})")
print(f"Passes with elastic      : {TTA_ELASTIC_PASSES}")
print(f"Merge strategies         : mean-softmax, mean-logits")
print("=" * 60)

In [ ]:
# =============================================================================
# Dihedral Group (D4) Transforms
# =============================================================================

def d4_forward(x, k):
    """
    Apply the k-th element of the dihedral group D4 to an NCHW tensor.

    k = 0..3 -> rot90 k times
    k = 4..7 -> horizontal flip, then rot90 (k - 4) times

    Flips and 90-degree rotations are exact memory permutations, so this
    introduces no interpolation error.
    """

    if k >= 4:
        x = torch.flip(x, dims=[3])

    r = k % 4

    if r:
        x = torch.rot90(x, r, dims=[2, 3])

    return x


def d4_inverse(x, k):
    """
    Undo d4_forward.

    forward is  rot^r . flip   (flip applied first), so the inverse is
    (rot^r . flip)^-1 = flip^-1 . rot^-r = flip . rot^-r
    """

    r = k % 4

    if r:
        x = torch.rot90(x, -r, dims=[2, 3])

    if k >= 4:
        x = torch.flip(x, dims=[3])

    return x


def _round_to_multiple(value, multiple=TTA_SIZE_MULTIPLE):
    """
    Round a spatial dimension to the nearest valid multiple for the encoder.
    """

    return max(multiple, int(round(value / multiple)) * multiple)


# ---------------------------------------------------------------------------
# Sanity check: every transform must round-trip exactly
# ---------------------------------------------------------------------------

_probe = torch.randn(2, 3, 64, 64)

for _k in TTA_D4_KS:

    _restored = d4_inverse(d4_forward(_probe, _k), _k)

    assert torch.equal(_probe, _restored), f"D4 round-trip failed for k={_k}"

# The 8 transforms must be distinct, otherwise TTA is averaging duplicates
_signatures = {
    d4_forward(_probe, _k).flatten()[:512].numpy().tobytes()
    for _k in TTA_D4_KS
}

assert len(_signatures) == len(TTA_D4_KS), "D4 transforms are not distinct"

del _probe, _signatures

print("D4 transforms verified: 8 distinct, all round-trip exactly.")

In [ ]:
# =============================================================================
# Elastic Deformation for TTA
# =============================================================================
#
# A TTA transform must be invertible: warp the input, predict, unwarp the
# logits, average. D4 inverts exactly. Elastic does not -- given a forward
# displacement field d, the inverse field d_inv must satisfy
#
#     d_inv(x) = -d(x + d_inv(x))
#
# which has no closed form. It is solved below by fixed-point iteration, the
# standard approach in deformable registration.


def _gaussian_kernel1d(sigma, device, dtype):

    radius = max(1, int(3.0 * sigma))

    x = torch.arange(-radius, radius + 1, device=device, dtype=dtype)

    k = torch.exp(-(x ** 2) / (2.0 * sigma ** 2))

    return k / k.sum()


def _smooth(field, sigma):
    """Separable Gaussian blur of an (N, 2, H, W) displacement field."""

    k = _gaussian_kernel1d(sigma, field.device, field.dtype)

    r = (k.numel() - 1) // 2
    c = field.shape[1]

    kx = k.view(1, 1, 1, -1).expand(c, 1, 1, -1)
    ky = k.view(1, 1, -1, 1).expand(c, 1, -1, 1)

    field = F.conv2d(F.pad(field, (r, r, 0, 0), mode="replicate"), kx, groups=c)
    field = F.conv2d(F.pad(field, (0, 0, r, r), mode="replicate"), ky, groups=c)

    return field


def _base_grid(n, h, w, device, dtype):
    """Identity sampling grid in align_corners=False convention."""

    ys = (torch.arange(h, device=device, dtype=dtype) + 0.5) / h * 2.0 - 1.0
    xs = (torch.arange(w, device=device, dtype=dtype) + 0.5) / w * 2.0 - 1.0

    gy, gx = torch.meshgrid(ys, xs, indexing="ij")

    return torch.stack([gx, gy], dim=-1).unsqueeze(0).expand(n, -1, -1, -1)


def make_elastic_field(n, h, w, device, generator,
                       alpha=None, sigma=None, dtype=torch.float32):
    """
    Smooth random displacement field, returned in NORMALISED grid units
    (the convention grid_sample expects), peak magnitude alpha pixels.
    """

    alpha = TTA_ELASTIC_ALPHA if alpha is None else alpha
    sigma = TTA_ELASTIC_SIGMA if sigma is None else sigma

    d = torch.randn(n, 2, h, w, device=device, dtype=dtype,
                    generator=generator)

    d = _smooth(d, sigma)

    # Normalise so the largest displacement is exactly alpha pixels,
    # otherwise the amplitude drifts with sigma and image size.
    peak = d.flatten(1).abs().amax(dim=1).clamp_min(1e-8)
    d = d / peak.view(n, 1, 1, 1) * alpha

    d_norm = torch.empty_like(d)
    d_norm[:, 0] = d[:, 0] * (2.0 / w)    # x channel
    d_norm[:, 1] = d[:, 1] * (2.0 / h)    # y channel

    return d_norm


def invert_elastic_field(d_norm, iters=None):
    """
    Fixed-point inversion. Set iters=1 to recover the common -d shortcut.
    Residual at alpha=12, sigma=12 on 256px: ~0.66px at 6 iters,
    ~0.23px at 10.
    """

    iters = TTA_ELASTIC_INV_ITERS if iters is None else iters

    n, _, h, w = d_norm.shape

    bg = _base_grid(n, h, w, d_norm.device, d_norm.dtype)

    inv = torch.zeros_like(d_norm)

    for _ in range(iters):

        g = bg + inv.permute(0, 2, 3, 1)

        inv = -F.grid_sample(
            d_norm, g,
            mode="bilinear",
            padding_mode="border",
            align_corners=False
        )

    return inv


def elastic_warp(x, d_norm):
    """
    Apply a displacement field to an (N, C, H, W) tensor.

    padding_mode="border" rather than "zeros": zero-padding would inject
    spurious background evidence at the frame edge, which then gets averaged
    into the TTA result as a confident background vote.
    """

    n, _, h, w = x.shape

    bg = _base_grid(n, h, w, x.device, x.dtype)

    g = bg + d_norm.permute(0, 2, 3, 1).to(x.dtype)

    return F.grid_sample(
        x, g,
        mode="bilinear",
        padding_mode="border",
        align_corners=False
    )


# ---------------------------------------------------------------------------
# Sanity check: quantify what the round trip costs on thin structures
# ---------------------------------------------------------------------------

_h = _w = 256

_probe = torch.zeros(1, 1, _h, _w)
_probe[:, :, ::17, :] = 1.0        # 1px lines: worst case for resampling
_probe[:, :, :, ::17] = 1.0

_gen = torch.Generator().manual_seed(TTA_ELASTIC_SEED)
_d = make_elastic_field(1, _h, _w, _probe.device, _gen)

_rt = elastic_warp(elastic_warp(_probe, _d), invert_elastic_field(_d))

_err_elastic = (_rt - _probe).abs().mean().item()

_err_d4 = (
    torch.rot90(torch.rot90(_probe, 1, [2, 3]), -1, [2, 3]) - _probe
).abs().mean().item()

print(f"Probe signal mass          : {_probe.mean().item():.4f}")
print(f"D4 round-trip error        : {_err_d4:.5f}  (exact)")
print(f"Elastic round-trip error   : {_err_elastic:.5f}")

for _s in TTA_SCALES:

    if _s == 1.0:
        continue

    _sz = (_round_to_multiple(_h * _s), _round_to_multiple(_w * _s))

    _down = F.interpolate(_probe, size=_sz, mode="bilinear",
                          align_corners=False)
    _up = F.interpolate(_down, size=(_h, _w), mode="bilinear",
                        align_corners=False)

    print(f"Scale {_s} round-trip error : "
          f"{(_up - _probe).abs().mean().item():.5f}")

assert _err_d4 == 0.0, "D4 must remain exact"

del _probe, _d, _rt


In [ ]:
# =============================================================================
# TTA Prediction
# =============================================================================

@torch.no_grad()
def predict_tta(
    model,
    image,
    d4_ks=None,
    scales=None,
    elastic_n=0,
    use_amp=True
):
    """
    Multi-scale + dihedral test-time augmentation.

    Parameters
    ----------
    image : torch.Tensor
        (N, C, H, W) float tensor, already on the target device.
    elastic_n : int
        Random elastic fields applied on top of each (scale, D4) view.
        0 reproduces the pure D4 x multi-scale behaviour exactly.

    Returns
    -------
    dict with
        "prob"   : (N, NUM_CLASSES, H, W) mean of softmax probabilities
        "logit"  : (N, NUM_CLASSES, H, W) mean of raw logits
        "passes" : number of forward passes actually performed
    """

    if d4_ks is None:
        d4_ks = TTA_D4_KS

    if scales is None:
        scales = TTA_SCALES

    model.eval()

    n, _, h, w = image.shape

    # 90/270-degree rotations only preserve shape on square inputs.
    # For non-square inputs fall back to the shape-preserving subgroup
    # {identity, rot180, hflip, vflip} = even k.
    ks = d4_ks if h == w else [k for k in d4_ks if k % 2 == 0]

    # Dedicated generator: elastic TTA must be reproducible across runs and
    # must not disturb the global RNG state.
    gen = None

    if elastic_n > 0:
        gen = torch.Generator(device=image.device)
        gen.manual_seed(TTA_ELASTIC_SEED)

    prob_sum = torch.zeros(
        n, NUM_CLASSES, h, w,
        device=image.device,
        dtype=torch.float32
    )

    logit_sum = torch.zeros_like(prob_sum)

    count = 0

    for scale in scales:

        if scale == 1.0:
            scaled = image

        else:
            size = (
                _round_to_multiple(h * scale),
                _round_to_multiple(w * scale)
            )

            scaled = F.interpolate(
                image,
                size=size,
                mode="bilinear",
                align_corners=False
            )

        for k in ks:

            augmented = d4_forward(scaled, k)

            # field = None is the undeformed view; the rest are elastic
            fields = [None]

            if elastic_n > 0:

                sh, sw = augmented.shape[-2:]

                for _ in range(elastic_n):
                    fields.append(
                        make_elastic_field(
                            augmented.shape[0], sh, sw,
                            augmented.device, gen
                        )
                    )

            for field in fields:

                view = augmented if field is None else elastic_warp(
                    augmented, field
                )

                with torch.amp.autocast(
                    "cuda",
                    enabled=use_amp and image.is_cuda
                ):
                    logits = model(view)[0]   # main head only

                logits = logits.float()

                # Unwarp in reverse order of application:
                # elastic (approximate) -> D4 (exact) -> resize (approximate)
                if field is not None:
                    logits = elastic_warp(
                        logits, invert_elastic_field(field)
                    )

                logits = d4_inverse(logits, k)

                if logits.shape[-2:] != (h, w):

                    logits = F.interpolate(
                        logits,
                        size=(h, w),
                        mode="bilinear",
                        align_corners=False
                    )

                logit_sum += logits
                prob_sum += torch.softmax(logits, dim=1)

                count += 1

    return {
        "prob": prob_sum / count,
        "logit": logit_sum / count,
        "passes": count,
    }

In [ ]:
# =============================================================================
# Test Evaluation: Baseline vs TTA (mean-softmax) vs TTA (mean-logits)
# =============================================================================

VARIANTS = ["baseline", "tta_prob", "tta_logit",
            "tta_elastic_prob", "tta_elastic_logit"]

VARIANT_LABELS = {
    "baseline":  "No TTA (1 pass)",
    "tta_prob":  "TTA mean-softmax",
    "tta_logit": "TTA mean-logits",
    "tta_elastic_prob":  "TTA+elastic softmax",
    "tta_elastic_logit": "TTA+elastic logits",
}


@torch.no_grad()
def test_model_tta(model, loader, device, epoch=EPOCHS):

    model.eval()

    # Collected rather than summed so absent-class nans can be skipped
    scores = {v: {"dice": [], "iou": []} for v in VARIANTS}

    pred_chunks = {v: [] for v in VARIANTS}
    target_chunks = []

    running_loss = 0.0

    for images, masks in tqdm(loader, desc="Testing (TTA)"):

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        # ---- Baseline: single un-augmented forward pass ----
        with torch.amp.autocast("cuda"):

            outputs = model(images)

            loss = hybrid_loss(outputs, masks, epoch)

        running_loss += float(loss.detach())

        # ---- TTA: D4 x multi-scale ----
        merged = predict_tta(model, images)

        # ---- TTA: same, plus elastic views (separate variant) ----
        merged_el = predict_tta(
            model, images, elastic_n=TTA_ELASTIC_N
        )

        scored = {
            "baseline":          outputs[0].float(),
            "tta_prob":          merged["prob"],
            "tta_logit":         merged["logit"],
            "tta_elastic_prob":  merged_el["prob"],
            "tta_elastic_logit": merged_el["logit"],
        }

        for v, pred in scored.items():

            dice, _ = dice_score(pred, masks)
            iou, _ = iou_score(pred, masks)

            scores[v]["dice"].append(dice)
            scores[v]["iou"].append(iou)

            pred_chunks[v].append(
                torch.argmax(pred, dim=1)
                .to(torch.uint8)
                .cpu()
                .numpy()
                .ravel()
            )

        target_chunks.append(
            masks.to(torch.uint8).cpu().numpy().ravel()
        )

        del outputs, merged, merged_el, scored

    targets_flat = np.concatenate(target_chunks)

    results = {}

    for v in VARIANTS:

        preds_flat = np.concatenate(pred_chunks[v])

        precision, recall, f1, _ = precision_recall_fscore_support(
            targets_flat,
            preds_flat,
            labels=list(range(NUM_CLASSES)),
            average=None,
            zero_division=0
        )

        results[v] = {
            "dice": float(np.nanmean(scores[v]["dice"])),
            "iou": float(np.nanmean(scores[v]["iou"])),
            "accuracy": accuracy_score(targets_flat, preds_flat),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

    results["baseline"]["loss"] = running_loss / len(loader)

    return results

In [ ]:
# =============================================================================
# Run TTA Test Evaluation
# =============================================================================

# Weights were restored by the "Load Best Checkpoint" cell above.

import time

_t0 = time.time()

tta_results = test_model_tta(
    model,
    test_loader,
    DEVICE
)

_elapsed = time.time() - _t0

print(f"\nEvaluation completed in {_elapsed:.1f}s "
      f"({_elapsed / max(1, len(test_loader)):.1f}s per image, "
      f"{TTA_NUM_PASSES + 1} forward passes each)")

In [ ]:
# =============================================================================
# Comparison Table
# =============================================================================

base = tta_results["baseline"]

_show = [v for v in VARIANTS if v != "baseline"]

print("=" * 88)
print("TEST SET RESULTS -- BASELINE vs TTA VARIANTS")
print("=" * 88)

header = f"{'Metric':<12}{'No TTA':>10}"
for v in _show:
    header += f"{VARIANT_LABELS[v]:>22}"
print(header)
print("-" * 88)

for key, name in [("dice", "Dice"), ("iou", "IoU"), ("accuracy", "Accuracy")]:

    row = f"{name:<12}{base[key]:>10.4f}"

    for v in _show:
        d = tta_results[v][key] - base[key]
        row += f"{tta_results[v][key]:>13.4f}{d:>+9.4f}"

    print(row)

print("-" * 88)
print(f"{'Test loss':<12}{base['loss']:>10.4f}")

print()
print("=" * 88)
print("PER-CLASS F1")
print("=" * 88)

print(header.replace("Metric", "Class"))
print("-" * 88)

for i, cls in enumerate(CLASS_NAMES):

    row = f"{cls:<12}{base['f1'][i]:>10.3f}"

    for v in _show:
        d = tta_results[v]["f1"][i] - base["f1"][i]
        row += f"{tta_results[v]['f1'][i]:>13.3f}{d:>+9.3f}"

    print(row)

print("=" * 88)

_best = max(VARIANTS, key=lambda v: tta_results[v]["dice"])
print(f"\nBest variant by mean Dice: {VARIANT_LABELS[_best]}")

# Elastic is only worth its ~3x cost if it beats plain TTA, not just baseline.
_plain = max(tta_results["tta_prob"]["dice"], tta_results["tta_logit"]["dice"])
_elast = max(tta_results["tta_elastic_prob"]["dice"],
             tta_results["tta_elastic_logit"]["dice"])

print(f"Elastic vs plain TTA     : {_elast - _plain:+.4f} Dice")
print("NOTE: with 8 test images the noise floor is roughly +/-0.02 Dice.")
print("      Treat anything smaller as no measurable difference.")



In [ ]:
# =============================================================================
# Per-Class F1 -- Grouped Bar Chart
# =============================================================================

x = np.arange(len(CLASS_NAMES))
width = 0.8 / len(VARIANTS)

colours = ["#888888", "#1f77b4", "#d62728", "#2ca02c", "#9467bd"]

plt.figure(figsize=(13, 5))

for i, v in enumerate(VARIANTS):

    offset = (i - (len(VARIANTS) - 1) / 2) * width

    plt.bar(
        x + offset,
        tta_results[v]["f1"],
        width,
        label=VARIANT_LABELS[v],
        color=colours[i % len(colours)]
    )

plt.xticks(x, CLASS_NAMES)
plt.ylabel("F1")
plt.title("Per-Class F1: Baseline vs TTA Variants")
plt.ylim(0, 1.05)
plt.grid(True, axis="y", alpha=0.3)
plt.legend(fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Qualitative Comparison: Baseline vs TTA
# =============================================================================

model.eval()

num_examples = 4

indices = np.random.choice(
    len(test_dataset),
    min(num_examples, len(test_dataset)),
    replace=False
)

fig, axes = plt.subplots(
    len(indices),
    5,
    figsize=(22, 4.6 * len(indices))
)

if len(indices) == 1:
    axes = axes[None, :]

with torch.no_grad():

    for row, idx in enumerate(indices):

        image, gt = test_dataset[idx]

        image_gpu = image.unsqueeze(0).to(DEVICE, non_blocking=True)

        # Baseline
        with torch.amp.autocast("cuda"):
            baseline_logits = model(image_gpu)[0]

        baseline_pred = torch.argmax(
            baseline_logits.float(), dim=1
        ).squeeze(0).cpu().numpy()

        # TTA
        merged = predict_tta(model, image_gpu)

        tta_pred = torch.argmax(
            merged["prob"], dim=1
        ).squeeze(0).cpu().numpy()

        gt = gt.numpy()

        # Where TTA and baseline disagree
        disagreement = (baseline_pred != tta_pred)

        panels = [
            (image.permute(1, 2, 0).numpy(), "Input"),
            (class_to_rgb(gt), "Ground Truth"),
            (class_to_rgb(baseline_pred), "No TTA"),
            (class_to_rgb(tta_pred), f"TTA ({merged['passes']} passes)"),
            (disagreement, f"Disagreement ({disagreement.mean() * 100:.2f}%)"),
        ]

        for col, (img, title) in enumerate(panels):

            ax = axes[row, col]

            if col == 4:
                ax.imshow(img, cmap="magma")
            else:
                ax.imshow(img)

            ax.set_title(title)
            ax.axis("off")

plt.tight_layout()
plt.show()

del merged, baseline_logits
torch.cuda.empty_cache()